## Logistic loss
Logistic loss measures how well a model’s predicted probabilities match the actual binary outcomes (0 or 1). It penalizes confident but incorrect predictions very heavily.

In [ ]:
import numpy as np

def logistic_loss(Y, y_pred_proba):
    loss = -(Y * np.log(y_pred_proba) + (1 - Y) * np.log(1 - y_pred_proba))
    return np.mean(loss)

## Calibration
Model calibration refers to how well a model’s predicted probabilities reflect the true likelihood of outcomes.

In a well-calibrated model, predictions match observed frequencies.

Calibration model is learning how to model: raw probability → true empirical probability

1. Train base classifier on training data
2. Predict probabilities on calibration data
3. Fit isotonic or logistic calibrator (X - predicted probs ; Y - true class)
4. Apply calibrator to test predictions
5. Evaluate using Brier score or calibration curve
6. Optionally compute classification metrics

In [ ]:
# Pipeline for calibrating a model
model = (...)
model.fit(X_train, Y_train)

y_pred_proba = model.predict_proba(X_calib)

calib_model = (...)
calib_model.fit(y_pred_proba.reshape(-1, 1), Y_calib)

pred_raw = model.predict_proba(X_test)
pred_calib = calib_model.predict_proba(pred_raw.reshape(-1,1))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

prob_true_raw, prob_pred_raw = calibration_curve(Y_test, raw_probs, n_bins=10, strategy='uniform')
prob_true_cal, prob_pred_cal = calibration_curve(Y_test, calibrated_probs, n_bins=10, strategy='uniform')

# Plot
plt.figure(figsize=(8, 6))
plt.plot(prob_pred_raw, prob_true_raw, marker='o', label='Raw model')
plt.plot(prob_pred_cal, prob_true_cal, marker='s', label='Calibrated model')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly calibrated')
plt.xlabel('Predicted probability')
plt.ylabel('Observed fraction of positives')
plt.title('Calibration Plot / Reliability Diagram')
plt.legend()
plt.grid(True)
plt.show()

## Confidence Intervals
Calculating confidence intervals using:
- CLT
- Markov's inequality
- Chebychev's ineugality
- Hoeffding's inequality
- Bennet's inequality

In [ ]:
# CLT
import numpy as np
from scipy import stats

def conf_interval(data, conf_level = 0.95):
    n = len(data)
    mean = np.mean(data)
    std = np.std(data, ddof=1)

    # CI using t-distribution
    alpha = 1 - conf_level
    t_crit = stats.t.ppf(1 - alpha/2, df=n-1)

    margin_error = t_crit * std / np.sqrt(n)
    ci = (mean - margin_error, mean + margin_error)
    print("Sample mean:", mean)
    print("95% CI:", ci)


Sample mean: 6.444444444444445
95% CI: (np.float64(5.163331035997686), np.float64(7.725557852891203))


In [16]:
# Markov's inequality
import numpy as np

def markov_conf_interval(data, conf_level=0.95):
    n = len(data)

    sample_mean = np.mean(data)
    alpha = 1 - conf_level
    upper_bound = sample_mean / alpha

    return upper_bound


In [17]:
def chebychev_conf_interval(data, conf_level=0.95):
    n = len(data)
    alfa = 1 - conf_level
    epsilon = np.sqrt(np.var(data)/(n*alfa))
    mean = np.mean(data)

    return (mean - epsilon, mean + epsilon)

In [18]:
import numpy as np

def hoeffding_conf_interval(data, a, b, conf_level=0.95):
    alpha = 1 - conf_level
    n = len(data)
    epsilon = (b - a) / np.sqrt(2 * n) * np.sqrt(np.log(2 / alpha))
    mean = np.mean(data)
    return (mean - epsilon, mean + epsilon)


In [19]:
import numpy as np

def bennet_conf_interval(data, conf_level=0.95):
    variance = np.var(data)
    alfa = 1 - conf_level
    n = len(data)

    epsilon = np.sqrt( (2 * variance * np.log(2/alfa)) / n ) + (5 * np.log(2/alfa)) / (3 * n)
    mean = np.mean(data)

    interval = (mean - epsilon, mean + epsilon)
    return interval

## Markov chains

In [28]:
# calculating Stationary Distribution
import numpy as np

def stationary_dist(P):
    eigenvals, eigenvecs = np.linalg.eig(P.T)

    # find the index of the eigenvalue that is 1
    index = np.argmin(np.abs(eigenvals - 1))

    stat = eigenvecs[:, index].flatten() # getting eigenvec with eigenval 1
    stat = stat / np.sum(stat) #normalizing vector
    return stat

In [22]:
# calculating if the MC is irreducible
import numpy as np
import networkx as nx

def is_irreducible(P):
    G = nx.DiGraph()
    n = P.shape[0]
    
    for i in range(n):
        for j in range(n):
            if P[i, j] > 0:
                G.add_edge(i, j)
    
    return nx.is_strongly_connected(G)

In [ ]:
# calculating if the MC is aperiodic
from math import gcd
from functools import reduce

def is_aperiodic(P, max_power=100):
    n = P.shape[0]
    powers = np.eye(n)
    return_times = []

    for k in range(1, max_power + 1):
        powers = powers @ P
        if powers[0, 0] > 0:  # return to state 0
            return_times.append(k)

    if len(return_times) == 0:
        return False

    period = reduce(gcd, return_times)
    return period == 1


In [ ]:
# calculating if the MC is reversible
def is_reversible(P, tol=1e-8):

    if not (is_irreducible(P) and is_aperiodic(P)): # if there is no stationary distribution
        return False
    
    P = np.array(P)
    n = P.shape[0]

    pi = stationary_dist(P)

    for i in range(n):
        for j in range(n):
            if not np.isclose(pi[i] * P[i, j], pi[j] * P[j, i], atol=tol):
                return False

    return True

In [ ]:
# calculating n-step transition matrix
import numpy as np

def n_step_transition(P, n):
    '''
    Calculating n-step transition matrix.
    P^n --> transitions after n steps.
    With increasing n, each row will converge to stationary dist.
    '''
    return np.linalg.matrix_power(P, n)

## Linear Congruential Generator LCG

In [ ]:
def linConGen(m, a, b, x0, n):
    '''A linear congruential sequence generator.
    
    Param m is the integer modulus to use in the generator.
    Param a is the integer multiplier.
    Param b is the integer increment.
    Param x0 is the integer seed.
    Param n is the integer number of desired pseudo-random numbers.
    
    Returns a list of n pseudo-random integer modulo m numbers.'''
    
    x = x0 # the seed
    retValue = [x % m]  # start the list with x=x0
    for i in range(2, n+1, 1):
        x = (a * x + b) % m # the generator, using modular arithmetic
        retValue.append(x) # append the new x to the list
    return retValue

## Accept-Reject Sampler

In [ ]:
def problem1_inversion(f, M, n_samples=1):

    if M is None: # finding max of a function
        from scipy.optimize import minimize_scalar
        res = minimize_scalar(lambda x: -f(x), bounds=(0, 1), method='bounded')
        M = f(res.x)

    result = []
    while (len(result) < n_samples):
        x1 = np.random.uniform(0,1)
        f_x = f(x1)
        x2 = np.random.uniform(0,1)
        if (x2 <= f_x/M):
            result.append(x1)
        else:
            continue
    return np.array(result)